In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
%cd /content

# 切到你的專案資料夾
%cd /content/drive/MyDrive/LSTM_PROGRAM

##################################
# GARCH-X，照計畫書的公式，但效果很差， VAR的 STD 是用 GARCH的SHAPE

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content
/content/drive/MyDrive/LSTM_PROGRAM


In [2]:
!mkdir -p ~/.ssh
!cp /content/drive/MyDrive/.ssh/id_ed25519* ~/.ssh/
!chmod 700 ~/.ssh
!chmod 600 ~/.ssh/id_ed25519

!eval "$(ssh-agent -s)" && ssh-add ~/.ssh/id_ed25519
!ssh-keyscan github.com >> ~/.ssh/known_hosts
!chmod 644 ~/.ssh/known_hosts

!ssh -T git@github.com

!git config --global user.email "joemi7878@gmail.com"
!git config --global user.name "joemi78"


Agent pid 10008
Identity added: /root/.ssh/id_ed25519 (joemi7878@gmail.com)
# github.com:22 SSH-2.0-ab54611
# github.com:22 SSH-2.0-ab54611
# github.com:22 SSH-2.0-ab54611
# github.com:22 SSH-2.0-ab54611
# github.com:22 SSH-2.0-ab54611
Hi joemi78! You've successfully authenticated, but GitHub does not provide shell access.


In [3]:
!pip install arch

!sudo apt-get update
!sudo apt-get install -y build-essential python3-dev r-base-dev

!pip install -U jedi
!pip install -U pip setuptools wheel

Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:2 https://cli.github.com/packages stable InRelease
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading packag

In [4]:
# =============================================================================
# ES1 GK-LSTM VaR-t  ── B 版本 (Walk-Forward Warm Update)
# =============================================================================
# 研究設計：
#   Stage 1 ── Base Model 訓練（2006-2021, batch training）
#   Stage 2 ── Walk-Forward 評估（2022-2025）
#               每日：predict → record → warm update → slide
#
# 目標變數 Y : gk_vol_daily  (GK 單日波動度，σ 尺度，不需再開根號)
# 輸入特徵 X : ES1_LN_RET, gk_vol_daily, garch_vol, VIX_CLOSE  (lookback=20)
# VaR      : t 分配, μ=0, shape clip(6,10), α=0.05 / 0.01
# Backtest : Kupiec UC + Christoffersen CC  (訓練期 & 滾動評估期)
# Baseline : GARCH 模型 (來自 es1_volatility_all_methods.csv)
# =============================================================================

import os, warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.stats import t as tdist, chi2, pearsonr

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, optimizers
warnings.filterwarnings('ignore')

# =============================================================================
# 0.  設定區  ── 所有參數集中在此，修改這裡即可
# =============================================================================

# ── 資料路徑 ──────────────────────────────────────────────────────────────────
PATH_DAY  = "./filtered_output/df_day.csv"
PATH_VOL  = "./real_volatility_multi_var/es1_volatility_all_methods.csv"
PATH_GK_DAILY  = "./real_volatility_multi_var/gk_vol_daily_var_es_daily.csv"

# ── 樣本切分 ──────────────────────────────────────────────────────────────────
TRAIN_START = "2006-01-01"
TRAIN_END   = "2021-12-31"
TEST_START  = "2022-01-01"


# ── 特徵 & 目標 ───────────────────────────────────────────────────────────────
# [改動1] TARGET_COL 由 gk_vol_daily → gk_daily_VaR_price_95
FEATURE_COLS = ['garch_vol', 'gk_daily_VaR_ret_95']
TARGET_COL   = 'gk_daily_VaR_ret_95'

OUTPUT_DIR = "./LSTM_B_diagnostics"
os.makedirs(OUTPUT_DIR, exist_ok=True)


# ── GPU 設定（有 GPU 自動加速；無 GPU 正常跑 CPU）────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f"✓ GPU: {len(gpus)} device(s)")
else:
    print("ℹ  No GPU, running on CPU")


# =============================================================================
# 1.  讀資料 & 建母資料表
# =============================================================================

def load_master(path_day: str, path_vol: str, path_gk_daily: str) -> pd.DataFrame:
    # """
    # 合併 df_day.csv 與 es1_volatility_all_methods.csv，
    # 以 DATE 對齊，篩選 2006 年之後，刪除特徵缺值。
    # """
    # ── df_day ──
    day = pd.read_csv(path_day)
    if 'DATE' not in day.columns:
        day = day.reset_index().rename(columns={day.columns[0]: 'DATE'})
    day['DATE'] = pd.to_datetime(day['DATE'], errors='coerce').dt.normalize()

    keep_day = ['DATE', 'ES1_LN_RET', 'ES1_CLOSE', 'ES1_LOW','ES1_VOLUME',
                'VIX_LN_RET', 'VIX_CLOSE']
    day = day[[c for c in keep_day if c in day.columns]].dropna(subset=['DATE'])

    # ── vol / garch / shape ──
    vol = pd.read_csv(path_vol)
    if 'DATE' not in vol.columns:
        vol = vol.reset_index().rename(columns={vol.columns[0]: 'DATE'})
    vol['DATE'] = pd.to_datetime(vol['DATE'], errors='coerce').dt.normalize()

    keep_vol = ['DATE', 'gk_vol_daily', 'garch_vol', 'shape']
    vol = vol[[c for c in keep_vol if c in vol.columns]].dropna(subset=['DATE'])

    # ── gk va95 ret / gk va95 price ──
    gk = pd.read_csv(path_gk_daily)
    if 'DATE' not in gk.columns:
        gk = gk.reset_index().rename(columns={gk.columns[0]: 'DATE'})
    gk['DATE'] = pd.to_datetime(gk['DATE'], errors='coerce').dt.normalize()

    keep_gk = ['DATE', 'VaR_ret_95', 'VaR_price_95']
    gk = gk[[c for c in keep_gk if c in gk.columns]].dropna(subset=['DATE'])
    gk = gk.rename(columns={
    'VaR_ret_95': 'gk_daily_VaR_ret_95',
    'VaR_price_95': 'gk_daily_VaR_price_95'
    })


    # ── merge ──
    df = pd.merge(day, vol, on='DATE', how='inner')
    df = pd.merge(df, gk, on='DATE', how='inner')

    df = df.sort_values('DATE').reset_index(drop=True)
    df = df[df['DATE'] >= TRAIN_START].reset_index(drop=True)
    df = df.dropna(subset=FEATURE_COLS + ['shape']).reset_index(drop=True)

    print(f"Master table: {len(df):,} rows  "
          f"({df['DATE'].min().date()} ~ {df['DATE'].max().date()})")
    return df


df = load_master(PATH_DAY, PATH_VOL, PATH_GK_DAILY)

for c in df.columns:
    print(c)


df_data = pd.DataFrame(df)

df_data_path = os.path.join(OUTPUT_DIR, "df_data.csv")
df_data.to_csv(df_data_path, index=False, encoding='utf-8-sig')

print(f"✓ Saved: {df_data_path}")


✓ GPU: 1 device(s)
Master table: 4,922 rows  (2006-01-03 ~ 2025-06-30)
DATE
ES1_LN_RET
ES1_CLOSE
ES1_LOW
ES1_VOLUME
VIX_LN_RET
VIX_CLOSE
gk_vol_daily
garch_vol
shape
gk_daily_VaR_ret_95
gk_daily_VaR_price_95
✓ Saved: ./LSTM_B_diagnostics/df_data.csv


In [5]:
# =============================================================================
# ES1 GK-LSTM — 直接預測 gk_daily_VaR_price_95  （B 版本 Walk-Forward）
# =============================================================================
# 研究設計：
#   Stage 1 ── Base Model 訓練（2006-2021, batch training）
#   Stage 2 ── Walk-Forward 評估（2022-2025）
#               每日：predict → record → warm update → slide
#
# 目標變數 Y : gk_daily_VaR_price_95  (價格層 VaR，直接預測)
# 輸入特徵 X : gk_vol_daily(log), garch_vol, VIX_CLOSE, ES1_LN_RET
# 三方比較   : GK Realized / GARCH / LSTM_GK  （訓練期 & 測試期）
# 隱含波動   : 由預測 VaR_price 反推 sigma_implied
# Backtest   : Kupiec UC + Christoffersen CC
#
# ── 相較於舊版 sp500_3 的主要改動 ──────────────────────────────────────────────
# [1] TARGET_COL: gk_vol_daily → gk_daily_VaR_price_95
# [2] 特徵縮放: X 仍用 log(gk_vol_daily)；Y 改用 log(-gk_daily_VaR_price_95/ES1_CLOSE_prev)
#     (因為 VaR_price 是正數且數值大，取 log 比例讓模型更容易學習；反縮放時還原)
# [3] 新增：sigma_implied 反推（由預測 VaR_price 逆推隱含波動）
# [4] 新增：三模型 VaR_price 比較圖（GK / GARCH / LSTM_GK）
# [5] 新增：隱含波動 vs gk_vol_daily vs garch_vol 比較圖
# [6] 新增：按 quintile 的 bias 診斷圖（VaR_price 版本）
# [7] VaR 回測邏輯改為 price-level violation（ES1_CLOSE < VaR_price_pred）
# [8] GARCH VaR_price 用相同公式建構，確保三方可比
# [9] 新增：描述統計 CSV + 隱含波動統計 CSV
# [10]輸出 df_data.csv 沿用既有母資料表路徑
# =============================================================================



# =============================================================================
# 0.  設定區  ── 所有參數集中在此
# =============================================================================

# ── 資料路徑 ──────────────────────────────────────────────────────────────────
# 直接使用已整合好的母資料表 df_data.csv（三份來源已完成 inner merge）
PATH_MASTER = "./LSTM_B_diagnostics/df_data.csv"   # ← 你的母資料表路徑

LOOKBACK = 20

# ── Base Model 超參數 ─────────────────────────────────────────────────────────
LSTM_UNITS  = 64
DROPOUT     = 0.2
LR_BASE     = 1e-3
EPOCHS      = 100
BATCH_SIZE  = 32
PATIENCE_ES = 20
PATIENCE_LR = 10

# ── Walk-Forward Warm Update 超參數 ──────────────────────────────────────────
LR_WARM     = 1e-4         # 比 Base LR 低 10×
WARM_EPOCHS = 1            # 每次 1 epoch，避免單日噪音過擬合
WARM_BATCH  = 1

# ── VaR 設定 ──────────────────────────────────────────────────────────────────
ALPHA_95 = 0.05
ALPHA_99 = 0.01
NU_CLIP  = (6, 10)         # shape 截尾範圍

# ── GPU ───────────────────────────────────────────────────────────────────────
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
    print(f"✓ GPU: {len(gpus)} device(s)")
else:
    print("ℹ  No GPU, running on CPU")


# =============================================================================
# 1.  讀母資料表
# =============================================================================

# df = pd.read_csv(PATH_MASTER, parse_dates=['DATE'])
df['DATE'] = pd.to_datetime(df['DATE'], errors='coerce').dt.normalize()
df = df.sort_values('DATE').reset_index(drop=True)
df = df[(df['DATE'] >= TRAIN_START)].reset_index(drop=True)
# df = df.dropna(subset=FEATURE_COLS + [TARGET_COL, 'shape', 'ES1_CLOSE']).reset_index(drop=True)

print(f"Master table: {len(df):,} rows  "
      f"({df['DATE'].min().date()} ~ {df['DATE'].max().date()})")
print(f"Columns: {list(df.columns)}")

# 描述統計表（需求書第十三節）
desc_cols = ['gk_vol_daily','garch_vol','gk_daily_VaR_ret_95',
             'gk_daily_VaR_price_95','ES1_LN_RET','VIX_CLOSE']
df_desc = df[desc_cols].describe().T.round(6)
df_desc.to_csv(os.path.join(OUTPUT_DIR, "desc_stats.csv"), encoding='utf-8-sig')
print("✓ Saved desc_stats.csv")


# =============================================================================
# 2. 特徵縮放 & 滑動視窗序列建構  (ret-target 版本)
# =============================================================================

train_mask = (df['DATE'] >= TRAIN_START) & (df['DATE'] <= TRAIN_END)

# -----------------------------------------------------------------------------
# A. X 原始值
# -----------------------------------------------------------------------------
X_raw = df[FEATURE_COLS].copy()

# 若某些欄位是正值且右偏，可先 log1p / log
# 目前你的 FEATURE_COLS = ['garch_vol', 'gk_daily_VaR_ret_95']
# garch_vol > 0，可以 log
# gk_daily_VaR_ret_95 < 0，不可 log
# if 'garch_vol' in X_raw.columns:
#     X_raw['garch_vol'] = np.log(np.clip(X_raw['garch_vol'].values, 1e-8, None))

X_raw = X_raw.values.astype(np.float32)

# -----------------------------------------------------------------------------
# B. Y 原始值（target = VaR_ret_95，通常為負值，不可取 log）
# -----------------------------------------------------------------------------
Y_raw_all = df[TARGET_COL].values.astype(np.float64)

# -----------------------------------------------------------------------------
# C. Robust scaling function
# -----------------------------------------------------------------------------
def fit_robust_scaler(arr_2d: np.ndarray):
    # """
    # arr_2d: shape = (n_samples, n_features)
    # 回傳 median 與 IQR
    # """
    med = np.median(arr_2d, axis=0)
    q25 = np.percentile(arr_2d, 25, axis=0)
    q75 = np.percentile(arr_2d, 75, axis=0)
    iqr = q75 - q25
    iqr = np.where(np.abs(iqr) < 1e-8, 1.0, iqr)
    return med, iqr

def transform_robust(arr_2d: np.ndarray, med: np.ndarray, iqr: np.ndarray, clip_value=5.0):
    z = (arr_2d - med) / iqr
    z = np.clip(z, -clip_value, clip_value)
    return z

def inverse_transform_robust(arr_2d: np.ndarray, med: np.ndarray, iqr: np.ndarray):
    return arr_2d * iqr + med

# -----------------------------------------------------------------------------
# D. Fit scaler only on TRAIN
# -----------------------------------------------------------------------------
X_train_raw = X_raw[train_mask]
Y_train_raw = Y_raw_all[train_mask].reshape(-1, 1)

x_med, x_iqr = fit_robust_scaler(X_train_raw)
y_med, y_iqr = fit_robust_scaler(Y_train_raw)

# -----------------------------------------------------------------------------
# E. Transform all data with TRAIN-fitted scaler
# -----------------------------------------------------------------------------
# 真正厚尾的是 gk_daily_VaR_ret_95
# 左尾超界比例大概是：
# clip=4：2.08%
# clip=5：1.24%
# clip=6：0.89%
# clip=8：0.44%
X_scaled_all = transform_robust(X_raw, x_med, x_iqr, clip_value=6.0).astype(np.float32)
Y_scaled_all = transform_robust(Y_raw_all.reshape(-1, 1), y_med, y_iqr, clip_value=6.0).ravel().astype(np.float32)


# -----------------------------------------------------------------------------
# F. inverse transform for prediction
# -----------------------------------------------------------------------------
def descale_var_ret(y_scaled: np.ndarray) -> np.ndarray:
    # """
    # 將模型輸出的 scaled VaR_ret 還原回原始 VaR_ret 尺度
    # """
    y_scaled = np.array(y_scaled, dtype=np.float64).reshape(-1, 1)
    y_raw = inverse_transform_robust(y_scaled, y_med, y_iqr).ravel()
    return y_raw

#############################################################################################################

# ── 滑動視窗序列建構 ──────────────────────────────────────────────────────────
def build_sequences(X_sc, Y_sc, dates, shapes, returns, closes, true_y_raw, lookback):
    # """
    # X[i] = X[i-lookback:i]  (前 lookback 天特徵)
    # Y[i] = Y[i]              (第 i 天目標值，已縮放)
    # """
    Xs, Ys = [], []
    s_dates, s_shapes, s_ret, s_close, s_true = [], [], [], [], []

    for i in range(lookback, len(X_sc)):
        Xs.append(X_sc[i - lookback: i])
        Ys.append(Y_sc[i])
        s_dates.append(dates[i])
        s_shapes.append(shapes[i])
        s_ret.append(returns[i])
        s_close.append(closes[i])
        s_true.append(true_y_raw[i])

    return (np.array(Xs, dtype=np.float32),
            np.array(Ys, dtype=np.float32),
            np.array(s_dates),
            np.array(s_shapes, dtype=np.float32),
            np.array(s_ret,    dtype=np.float32),
            np.array(s_close,  dtype=np.float32),
            np.array(s_true,   dtype=np.float32))


(X_seq, Y_seq, seq_dates, seq_shapes,
 seq_ret, seq_close, seq_true) = build_sequences(
    X_scaled_all, Y_scaled_all,
    df['DATE'].values, df['shape'].values,
    df['ES1_LN_RET'].values, df['ES1_CLOSE'].values,
    Y_raw_all,LOOKBACK
)

# ── 訓練 / 測試切分 ───────────────────────────────────────────────────────────
tr_mask = pd.to_datetime(seq_dates) <= pd.Timestamp(TRAIN_END)
te_mask = pd.to_datetime(seq_dates) >= pd.Timestamp(TEST_START)

X_train, Y_train    = X_seq[tr_mask], Y_seq[tr_mask]
X_test,  Y_test     = X_seq[te_mask], Y_seq[te_mask]
dates_tr, dates_te  = pd.to_datetime(seq_dates[tr_mask]), pd.to_datetime(seq_dates[te_mask])
shape_tr, shape_te  = seq_shapes[tr_mask], seq_shapes[te_mask]
ret_tr,   ret_te    = seq_ret[tr_mask],    seq_ret[te_mask]
close_tr, close_te  = seq_close[tr_mask],  seq_close[te_mask]
true_y_tr, true_y_te = seq_true[tr_mask],  seq_true[te_mask]  # gk_daily_VaR_ret_95

n_features = X_train.shape[2]
print(f"\nTrain: {X_train.shape}  ({dates_tr[0].date()} ~ {dates_tr[-1].date()})")
print(f"Test : {X_test.shape}   ({dates_te[0].date()} ~ {dates_te[-1].date()})")
print(f"n_features = {n_features}")


# =============================================================================
# 3.  LSTM 模型建構（tensorflow.keras）
# =============================================================================

def build_lstm_model(lookback, n_feat, lstm_units=LSTM_UNITS,
                     dropout=DROPOUT, lr=LR_BASE):
    # """
    # Input → LSTM → Dropout → Dense(linear)
    # 輸出為  VaR95_ret，線性輸出層（無 softplus）
    # 因為 Y 已 經過4分位
    # 損失：Huber（對尾端 VaR 極端值更穩健）
    # """
    inp = layers.Input(shape=(lookback, n_feat), name="seq_input")
    x   = layers.LSTM(lstm_units, return_sequences=False, name="lstm_1")(inp)
    x   = layers.Dropout(dropout, name="dropout")(x)
    out = layers.Dense(1, activation='linear', name="var_price_output")(x)

    m = models.Model(inp, out, name="GK_LSTM_VaR_Price")
    m.compile(
        optimizer=optimizers.Adam(learning_rate=lr),
        loss=tf.keras.losses.Huber(delta=1.0)  # Huber 對極端 VaR 值更穩健
    )
    return m


# =============================================================================
# 4.  Stage 1：Base Model 訓練（2006–2021）
# =============================================================================

print("\n" + "="*60)
print("STAGE 1: Base Model Training (2006-2021)")
print("="*60)

base_model = build_lstm_model(LOOKBACK, n_features, lr=LR_BASE)
base_model.summary()

cb_list = [
    callbacks.EarlyStopping(
        monitor='val_loss', patience=PATIENCE_ES,
        restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(
        monitor='val_loss', factor=0.5,
        patience=PATIENCE_LR, min_lr=1e-6, verbose=1),
    callbacks.ModelCheckpoint(
        filepath=os.path.join(OUTPUT_DIR, "base_model_best.keras"),
        monitor='val_loss', save_best_only=True, verbose=0),
]

history = base_model.fit(
    X_train, Y_train,
    validation_split=0.1,   # 末 10% 作驗證集（時序保留）
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    shuffle=False,           # 時間序列嚴禁打亂
    callbacks=cb_list,
    verbose=1
)
print(f"✓ Base model trained: {len(history.history['loss'])} epochs")

# ── 訓練期 in-sample 預測 ────────────────────────────────────────────────────
pred_tr_s = base_model.predict(X_train, verbose=0).ravel()
pred_tr   = descale_var_ret(pred_tr_s)    # gk_daily_VaR_ret_95 預測值（原始 ret 尺度，通常為負）

rmse_tr = float(np.sqrt(mean_squared_error(true_y_tr, pred_tr)))
mae_tr  = float(mean_absolute_error(true_y_tr, pred_tr))
corr_tr = float(pearsonr(pred_tr, true_y_tr)[0])
r2_tr   = float(r2_score(true_y_tr, pred_tr))

print(f"[Train] RMSE={rmse_tr:.6f}  MAE={mae_tr:.6f}  Corr={corr_tr:.4f}  R2={r2_tr:.4f}")


✓ GPU: 1 device(s)
Master table: 4,922 rows  (2006-01-03 ~ 2025-06-30)
Columns: ['DATE', 'ES1_LN_RET', 'ES1_CLOSE', 'ES1_LOW', 'ES1_VOLUME', 'VIX_LN_RET', 'VIX_CLOSE', 'gk_vol_daily', 'garch_vol', 'shape', 'gk_daily_VaR_ret_95', 'gk_daily_VaR_price_95']
✓ Saved desc_stats.csv

Train: (4025, 20, 2)  (2006-01-31 ~ 2021-12-31)
Test : (877, 20, 2)   (2022-01-03 ~ 2025-06-30)
n_features = 2

STAGE 1: Base Model Training (2006-2021)


Model: "GK_LSTM_VaR_Price"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ seq_input (InputLayer)          │ (None, 20, 2)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        17,152 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ var_price_output (Dense)        │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,217 (67.25 KB)

 Trainable params: 17,217 (67.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 4s 8ms/step - loss: 0.2404 - val_loss: 0.2053 - learning_rate: 0.0010
Epoch 2/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 3s 9ms/step - loss: 0.1916 - val_loss: 0.1987 - learning_rate: 0.0010
Epoch 3/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.1807 - val_loss: 0.1944 - learning_rate: 0.0010
Epoch 4/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.1753 - val_loss: 0.1904 - learning_rate: 0.0010
Epoch 5/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 0.1736 - val_loss: 0.1877 - learning_rate: 0.0010
Epoch 6/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.1729 - val_loss: 0.1870 - learning_rate: 0.0010
Epoch 7/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.1714 - val_loss: 0.1856 - learning_rate: 0.0010
Epoch 8/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.1723 - val_loss: 0.1837 - learning_rate: 0.0010
Epoch 9/100
114/114 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.1729 - val_loss: 0.1826 - learning_rate: 0.0010
E

In [6]:
# ── Frozen 測試集預測（A版基準） ──────────────────────────────────────────────
pred_frozen_s = base_model.predict(X_test, verbose=0).ravel()
pred_frozen   = descale_var_ret(pred_frozen_s)

rmse_frozen = float(np.sqrt(mean_squared_error(true_y_te, pred_frozen)))
mae_frozen  = float(mean_absolute_error(true_y_te, pred_frozen))
corr_frozen = float(pearsonr(pred_frozen, true_y_te)[0])
print(f"[Frozen] RMSE={rmse_frozen:.2f}  MAE={mae_frozen:.2f}  Corr={corr_frozen:.4f}")

base_model.save(os.path.join(OUTPUT_DIR, "base_model_final.keras"))
print("✓ Base model saved.")


# =============================================================================
# 5.  Stage 2：Walk-Forward Warm Update（2022–2025）
# =============================================================================

print("\n" + "="*60)
print("STAGE 2: Walk-Forward Warm Update (2022-2025)")
print(f"  Warm LR={LR_WARM}, Epochs={WARM_EPOCHS}, Batch={WARM_BATCH}")
print("="*60)

warm_model = build_lstm_model(LOOKBACK, n_features, lr=LR_WARM)
warm_model.set_weights(base_model.get_weights())

# 驗證權重複製
_cb = base_model.predict(X_test[:3], verbose=0).ravel()
_cw = warm_model.predict(X_test[:3], verbose=0).ravel()
assert np.allclose(_cb, _cw, atol=1e-5), "Weight copy failed!"
print("✓ Warm model weights copied.")

full_dates     = df['DATE'].values
test_pos_start = np.searchsorted(full_dates, np.datetime64(TEST_START))

# walk-forward 結果容器（ret-target 版本）
walk_pred   = []   # LSTM 預測的 gk_daily_VaR_ret_95（更新前記錄）
walk_dates  = []
walk_shapes = []
walk_ret    = []   # 真實 ES1 日報酬（之後做 violation 會用到）
walk_close  = []   # ES1_CLOSE（若之後仍要轉成 price-level 圖可用）
walk_true   = []   # 真實 gk_daily_VaR_ret_95
walk_garch  = []   # 同期 garch_vol（baseline / 轉 VaR_ret 用）
walk_gk_vol = []   # 同期 gk_vol_daily（診斷或比較用）

n_walk = len(full_dates) - test_pos_start
print(f"Walk-forward: {n_walk} days")

for i, pos in enumerate(range(test_pos_start, len(df))):
    if pos < LOOKBACK:
        continue

    # 用前 LOOKBACK 天特徵預測當天 target
    window  = X_scaled_all[pos - LOOKBACK: pos]
    x_input = window.reshape(1, LOOKBACK, n_features).astype(np.float32)

    # Step A: 先預測（必須在 warm update 前）
    y_pred_s   = warm_model.predict(x_input, verbose=0).ravel()[0]   # scaled VaR_ret
    y_pred_ret = float(descale_var_ret(np.array([y_pred_s]))[0])     # 還原成原始 VaR_ret

    # Step B: 記錄 prediction / truth / 其他欄位
    walk_pred.append(y_pred_ret)
    walk_dates.append(full_dates[pos])
    walk_shapes.append(float(df['shape'].values[pos]))
    walk_ret.append(float(df['ES1_LN_RET'].values[pos]))
    walk_close.append(float(df['ES1_CLOSE'].values[pos]))
    walk_true.append(float(Y_raw_all[pos]))          # 真實 gk_daily_VaR_ret_95
    walk_garch.append(float(df['garch_vol'].values[pos]))
    walk_gk_vol.append(float(df['gk_vol_daily'].values[pos]))

    # Step C: 用當天真實 scaled target 做 warm update
    y_true_s = float(Y_scaled_all[pos])
    warm_model.fit(
        x_input,
        np.array([[y_true_s]], dtype=np.float32),
        epochs=WARM_EPOCHS,
        batch_size=WARM_BATCH,
        verbose=0,
        shuffle=False
    )

    if (i + 1) % 200 == 0 or (i + 1) == n_walk:
        print(f"  Walk-forward: {i+1:4d}/{n_walk} done")

# 整理 walk-forward 結果
walk_pred   = np.array(walk_pred,   dtype=np.float64)   # 預測 VaR_ret_95
walk_dates  = pd.to_datetime(walk_dates)
walk_shapes = np.array(walk_shapes, dtype=np.float64)
walk_ret    = np.array(walk_ret,    dtype=np.float64)   # 真實 ES1_LN_RET
walk_close  = np.array(walk_close,  dtype=np.float64)
walk_true   = np.array(walk_true,   dtype=np.float64)   # 真實 VaR_ret_95
walk_garch  = np.array(walk_garch,  dtype=np.float64)
walk_gk_vol = np.array(walk_gk_vol, dtype=np.float64)

# ret-space 預測誤差
rmse_wf = float(np.sqrt(mean_squared_error(walk_true, walk_pred)))
mae_wf  = float(mean_absolute_error(walk_true, walk_pred))
corr_wf = float(pearsonr(walk_pred, walk_true)[0])

print(f"\n[Walk-Fwd B | VaR_ret] RMSE={rmse_wf:.6f}  MAE={mae_wf:.6f}  Corr={corr_wf:.4f}")
print(f"[Frozen   A | VaR_ret] RMSE={rmse_frozen:.6f}  MAE={mae_frozen:.6f}  Corr={corr_frozen:.4f}")

warm_model.save(os.path.join(OUTPUT_DIR, "warm_model_final.keras"))
print("✓ Warm model saved.")


# =============================================================================
# 6.  GARCH & GK Realized VaR_price 建構（三方比較基準）
# =============================================================================

def build_var_t_ret(sigma, shape, alpha=ALPHA_95):
    """
    VaR return = sigma × s × t_ν^{-1}(α)  (負值)
    """
    nu = np.clip(shape, *NU_CLIP)
    s  = np.sqrt((nu - 2.0) / nu)
    q  = tdist.ppf(alpha, df=nu)
    return sigma * s * q

def var_ret_to_price(close_prev, var_ret):
    """
    VaR_price = P_{t-1} × exp(VaR_ret)
    """
    return np.array(close_prev, dtype=np.float64) * np.exp(np.array(var_ret, dtype=np.float64))

df_idx = df.set_index('DATE')

# -----------------------------------------------------------------------------
# 取 train / test 對應欄位
# -----------------------------------------------------------------------------
gk_vol_tr_arr    = np.array([df_idx.loc[d, 'gk_vol_daily']         for d in dates_tr], dtype=np.float64)
garch_vol_tr_arr = np.array([df_idx.loc[d, 'garch_vol']            for d in dates_tr], dtype=np.float64)
shape_tr_arr     = np.array([df_idx.loc[d, 'shape']                for d in dates_tr], dtype=np.float64)
gk_var_ret_tr    = np.array([df_idx.loc[d, 'gk_daily_VaR_ret_95']  for d in dates_tr], dtype=np.float64)

gk_var_ret_te    = np.array([df_idx.loc[d, 'gk_daily_VaR_ret_95']  for d in walk_dates], dtype=np.float64)

# -----------------------------------------------------------------------------
# 前一日收盤：轉 VaR_price 用
# -----------------------------------------------------------------------------
def get_prev_close(dates_arr):
    all_dates = df['DATE'].values
    result = np.full(len(dates_arr), np.nan)
    for i, d in enumerate(dates_arr):
        pos = np.searchsorted(all_dates, d)
        if pos > 0:
            result[i] = df['ES1_CLOSE'].values[pos - 1]
    return result

prev_close_tr = get_prev_close(dates_tr.values)
prev_close_te = get_prev_close(dates_te.values)
prev_close_wf = get_prev_close(walk_dates.values)

# -----------------------------------------------------------------------------
# 三方 VaR_ret
# -----------------------------------------------------------------------------
# GK benchmark
gk_var_ret_train = gk_var_ret_tr
gk_var_ret_test  = gk_var_ret_te

# GARCH baseline
garch_var_ret_tr = build_var_t_ret(garch_vol_tr_arr, shape_tr_arr, ALPHA_95)
garch_var_ret_te = build_var_t_ret(walk_garch,       walk_shapes,  ALPHA_95)

# LSTM
lstm_var_ret_tr  = pred_tr
lstm_var_ret_frz = pred_frozen
lstm_var_ret_wf  = walk_pred

# -----------------------------------------------------------------------------
# 三方 VaR_price（只作圖或額外展示用）
# -----------------------------------------------------------------------------
gk_var_price_tr    = var_ret_to_price(prev_close_tr, gk_var_ret_train)
garch_var_price_tr = var_ret_to_price(prev_close_tr, garch_var_ret_tr)
lstm_var_price_tr  = var_ret_to_price(prev_close_tr, lstm_var_ret_tr)

gk_var_price_te    = var_ret_to_price(prev_close_wf, gk_var_ret_test)
garch_var_price_te = var_ret_to_price(prev_close_wf, garch_var_ret_te)
lstm_var_price_wf  = var_ret_to_price(prev_close_wf, lstm_var_ret_wf)
lstm_var_price_frz = var_ret_to_price(prev_close_te, lstm_var_ret_frz)


# =============================================================================
# 7.  [新增] 隱含波動反推（sigma_implied）
#     VaR_price = P_{t-1} × exp(VaR_ret)
#     VaR_ret   = sigma × s × q_α
#     → sigma_implied = VaR_ret / (s × q_α)  = abs(log(VaR_price/P_{t-1})) / (s × |q_α|)
# =============================================================================

# =============================================================================
# 7.  隱含波動反推（sigma_implied）
#     由 VaR_ret 直接反推 sigma
# =============================================================================

def implied_sigma_from_ret(var_ret, shape, alpha=ALPHA_95):
    # """
    # 由 VaR_ret 反推隱含波動 sigma_implied
    # VaR_ret = sigma * s * q_alpha   (q_alpha < 0)
    # => sigma = |VaR_ret| / |s * q_alpha|
    # """
    nu = np.clip(np.array(shape, dtype=np.float64), *NU_CLIP)
    s  = np.sqrt((nu - 2.0) / nu)
    q  = tdist.ppf(alpha, df=nu)   # 負值
    sigma = np.abs(np.array(var_ret, dtype=np.float64)) / np.abs(s * q)
    return sigma

# 訓練期
sigma_implied_tr  = implied_sigma_from_ret(lstm_var_ret_tr,  shape_tr_arr)

# 測試期
sigma_implied_frz = implied_sigma_from_ret(lstm_var_ret_frz, shape_te)
sigma_implied_wf  = implied_sigma_from_ret(lstm_var_ret_wf,  walk_shapes)

print(f"\n[Implied Sigma]")
print(f"  Train mean={np.nanmean(sigma_implied_tr):.6f}")
print(f"  Frozen mean={np.nanmean(sigma_implied_frz):.6f}")
print(f"  WalkFwd mean={np.nanmean(sigma_implied_wf):.6f}")
print(f"  GK mean={np.nanmean(walk_gk_vol):.6f}")
print(f"  GARCH mean={np.nanmean(walk_garch):.6f}")


# =============================================================================
# 8.  VaR 回測 — Return-Level Violation
#     violation: ES1_LN_RET_t < predicted_VaR_ret_t
# =============================================================================

def kupiec_test(viol, alpha=ALPHA_95):
    v = np.asarray(viol, dtype=int)
    n = len(v)
    x = int(v.sum())
    eps = 1e-12
    phat = np.clip(x / n, eps, 1 - eps)
    ac   = np.clip(alpha, eps, 1 - eps)
    LR_uc = -2 * (
        (n - x) * np.log(1 - ac) + x * np.log(ac)
        - (n - x) * np.log(1 - phat) - x * np.log(phat)
    )
    return dict(alpha=alpha, n=n, x=x, viol_rate=float(x / n),
                LR_uc=float(LR_uc), p_uc=float(1 - chi2.cdf(LR_uc, 1)))

def christoffersen_cc(viol, alpha=ALPHA_95):
    v = np.asarray(viol, dtype=int)
    uc = kupiec_test(v, alpha)

    vl, vn = v[:-1], v[1:]
    n00 = int(np.sum((vl == 0) & (vn == 0)))
    n01 = int(np.sum((vl == 0) & (vn == 1)))
    n10 = int(np.sum((vl == 1) & (vn == 0)))
    n11 = int(np.sum((vl == 1) & (vn == 1)))

    eps = 1e-12
    pi01 = np.clip(n01 / max(n00 + n01, 1), eps, 1 - eps)
    pi11 = np.clip(n11 / max(n10 + n11, 1), eps, 1 - eps)
    pi   = np.clip((n01 + n11) / max(n00 + n01 + n10 + n11, 1), eps, 1 - eps)

    LR_ind = -2 * (
        (n00 + n10) * np.log(1 - pi) + (n01 + n11) * np.log(pi)
        - n00 * np.log(1 - pi01) - n01 * np.log(pi01)
        - n10 * np.log(1 - pi11) - n11 * np.log(pi11)
    )
    LR_cc = uc['LR_uc'] + LR_ind

    return dict(**uc, n00=n00, n01=n01, n10=n10, n11=n11,
                LR_ind=float(LR_ind), p_ind=float(1 - chi2.cdf(LR_ind, 1)),
                LR_cc=float(LR_cc),   p_cc=float(1 - chi2.cdf(LR_cc, 2)))

def run_bt(viol, alpha, label):
    k = kupiec_test(viol, alpha)
    c = christoffersen_cc(viol, alpha)
    print(f"  [{label:32s}] N={k['n']} Viol={k['x']} ({k['viol_rate']*100:.2f}%)  "
          f"UC={'✓' if k['p_uc']>=0.05 else '✗'}(p={k['p_uc']:.4f})  "
          f"CC={'✓' if c['p_cc']>=0.05 else '✗'}(p={c['p_cc']:.4f})  "
          f"IND p={c['p_ind']:.4f}")
    return k, c

# -----------------------------------------------------------------------------
# Train violations
# -----------------------------------------------------------------------------
viol_gk_tr    = (ret_tr   < gk_var_ret_train).astype(int)
viol_garch_tr = (ret_tr   < garch_var_ret_tr).astype(int)
viol_lstm_tr  = (ret_tr   < lstm_var_ret_tr).astype(int)

# -----------------------------------------------------------------------------
# Test violations
# -----------------------------------------------------------------------------
viol_gk_te    = (walk_ret < gk_var_ret_test).astype(int)
viol_garch_te = (walk_ret < garch_var_ret_te).astype(int)
viol_lstm_frz = (ret_te   < lstm_var_ret_frz).astype(int)
viol_lstm_wf  = (walk_ret < lstm_var_ret_wf).astype(int)

print("\n" + "="*75)
print("VaR(95%) Backtest — Return-Level Violations")
print("="*75)

print("  [TRAIN 2006-2021]")
k_gk_tr, c_gk_tr   = run_bt(viol_gk_tr,    ALPHA_95, "GK Realized")
k_ga_tr, c_ga_tr   = run_bt(viol_garch_tr, ALPHA_95, "GARCH baseline")
k_ls_tr, c_ls_tr   = run_bt(viol_lstm_tr,  ALPHA_95, "LSTM_GK Base")

print("  [TEST 2022-2025]")
k_gk_te, c_gk_te   = run_bt(viol_gk_te,    ALPHA_95, "GK Realized")
k_ga_te, c_ga_te   = run_bt(viol_garch_te, ALPHA_95, "GARCH baseline")
k_ls_frz, c_ls_frz = run_bt(viol_lstm_frz, ALPHA_95, "LSTM Frozen")
k_ls_wf,  c_ls_wf  = run_bt(viol_lstm_wf,  ALPHA_95, "LSTM WalkFwd")

[Frozen] RMSE=0.01  MAE=0.00  Corr=0.7062
✓ Base model saved.

STAGE 2: Walk-Forward Warm Update (2022-2025)
  Warm LR=0.0001, Epochs=1, Batch=1
✓ Warm model weights copied.
Walk-forward: 877 days
  Walk-forward:  200/877 done
  Walk-forward:  400/877 done
  Walk-forward:  600/877 done
  Walk-forward:  800/877 done
  Walk-forward:  877/877 done

[Walk-Fwd B | VaR_ret] RMSE=0.006961  MAE=0.004612  Corr=0.7064
[Frozen   A | VaR_ret] RMSE=0.007006  MAE=0.004587  Corr=0.7062
✓ Warm model saved.

[Implied Sigma]
  Train mean=0.008687
  Frozen mean=0.009086
  WalkFwd mean=0.009303
  GK mean=0.009731
  GARCH mean=0.010947

VaR(95%) Backtest — Return-Level Violations
  [TRAIN 2006-2021]
  [GK Realized                     ] N=4025 Viol=175 (4.35%)  UC=✓(p=0.0524)  CC=✓(p=0.1248)  IND p=0.5274
  [GARCH baseline                  ] N=4025 Viol=233 (5.79%)  UC=✗(p=0.0249)  CC=✓(p=0.0800)  IND p=0.8838
  [LSTM_GK Base                    ] N=4025 Viol=306 (7.60%)  UC=✗(p=0.0000)  CC=✗(p=0.0000)  IND 

In [7]:
# =============================================================================
# 9.  CSV 輸出（ret-target 版本）
# =============================================================================

def make_row(k, c, split, model, level='95'):
    return {
        'model': model,
        'split': split,
        'level': level,
        'N': k['n'],
        'violations': k['x'],
        'viol_rate': round(k['viol_rate'], 6),
        'kupiec_LR': round(k['LR_uc'], 4),
        'kupiec_p': round(k['p_uc'], 4),
        'cc_LR': round(c['LR_cc'], 4),
        'cc_p': round(c['p_cc'], 4),
        'ind_p': round(c['p_ind'], 4),
        'n00': c['n00'],
        'n01': c['n01'],
        'n10': c['n10'],
        'n11': c['n11'],
        'uc_pass': k['p_uc'] >= 0.05,
        'cc_pass': c['p_cc'] >= 0.05
    }

# -----------------------------------------------------------------------------
# 9.1 回測彙總（Return-Level Violations）
# -----------------------------------------------------------------------------
df_bt = pd.DataFrame([
    make_row(k_gk_tr,  c_gk_tr,  'train', 'GK_Realized'),
    make_row(k_ga_tr,  c_ga_tr,  'train', 'GARCH'),
    make_row(k_ls_tr,  c_ls_tr,  'train', 'LSTM_GK_Base'),
    make_row(k_gk_te,  c_gk_te,  'test',  'GK_Realized'),
    make_row(k_ga_te,  c_ga_te,  'test',  'GARCH'),
    make_row(k_ls_frz, c_ls_frz, 'test',  'LSTM_GK_Frozen'),
    make_row(k_ls_wf,  c_ls_wf,  'test',  'LSTM_GK_WalkFwd'),
])
df_bt.to_csv(os.path.join(OUTPUT_DIR, "backtest_summary.csv"), index=False, encoding='utf-8-sig')
print("✓ Saved backtest_summary.csv")


# -----------------------------------------------------------------------------
# 9.2 預測誤差彙總（VaR_ret_95）
# -----------------------------------------------------------------------------
df_vm = pd.DataFrame([
    {
        'model': 'LSTM_GK_Train',
        'split': 'train',
        'target': 'VaR_ret_95',
        'RMSE': rmse_tr,
        'MAE': mae_tr,
        'Corr': corr_tr
    },
    {
        'model': 'LSTM_GK_Frozen',
        'split': 'frozen',
        'target': 'VaR_ret_95',
        'RMSE': rmse_frozen,
        'MAE': mae_frozen,
        'Corr': corr_frozen
    },
    {
        'model': 'LSTM_GK_WalkFwd',
        'split': 'wf',
        'target': 'VaR_ret_95',
        'RMSE': rmse_wf,
        'MAE': mae_wf,
        'Corr': corr_wf
    },
])
df_vm.to_csv(os.path.join(OUTPUT_DIR, "forecast_metrics.csv"), index=False, encoding='utf-8-sig')
print("✓ Saved forecast_metrics.csv")


# -----------------------------------------------------------------------------
# 9.3 三模型 VaR_ret / VaR_price 描述統計
# -----------------------------------------------------------------------------
def desc_arr(arr, name):
    a = pd.Series(arr).dropna()
    return {
        'name': name,
        'mean': a.mean(),
        'median': a.median(),
        'std': a.std(),
        'min': a.min(),
        'max': a.max()
    }

# VaR_ret 描述統計（主表）
df_var_ret_stats = pd.DataFrame([
    desc_arr(gk_var_ret_train, 'GK_Realized_VaR_ret_train'),
    desc_arr(garch_var_ret_tr, 'GARCH_VaR_ret_train'),
    desc_arr(lstm_var_ret_tr,  'LSTM_GK_VaR_ret_train'),

    desc_arr(gk_var_ret_test,  'GK_Realized_VaR_ret_test'),
    desc_arr(garch_var_ret_te, 'GARCH_VaR_ret_test'),
    desc_arr(lstm_var_ret_frz, 'LSTM_GK_Frozen_VaR_ret_test'),
    desc_arr(lstm_var_ret_wf,  'LSTM_GK_WalkFwd_VaR_ret_test'),
])
df_var_ret_stats.to_csv(os.path.join(OUTPUT_DIR, "var_ret_stats.csv"), index=False, encoding='utf-8-sig')
print("✓ Saved var_ret_stats.csv")

# VaR_price 描述統計（由 ret 轉出，展示用途）
df_var_price_stats = pd.DataFrame([
    desc_arr(gk_var_price_tr,    'GK_Realized_VaR_price_train'),
    desc_arr(garch_var_price_tr, 'GARCH_VaR_price_train'),
    desc_arr(lstm_var_price_tr,  'LSTM_GK_VaR_price_train'),

    desc_arr(gk_var_price_te,    'GK_Realized_VaR_price_test'),
    desc_arr(garch_var_price_te, 'GARCH_VaR_price_test'),
    desc_arr(lstm_var_price_frz, 'LSTM_GK_Frozen_VaR_price_test'),
    desc_arr(lstm_var_price_wf,  'LSTM_GK_WalkFwd_VaR_price_test'),
])
df_var_price_stats.to_csv(os.path.join(OUTPUT_DIR, "var_price_stats.csv"), index=False, encoding='utf-8-sig')
print("✓ Saved var_price_stats.csv")


# -----------------------------------------------------------------------------
# 9.4 隱含波動統計表（由 VaR_ret 反推）
# -----------------------------------------------------------------------------
df_sig_stats = pd.DataFrame([
    desc_arr(sigma_implied_tr,  'LSTM_implied_sigma_from_ret_train'),
    desc_arr(gk_vol_tr_arr,     'GK_vol_train'),
    desc_arr(garch_vol_tr_arr,  'GARCH_vol_train'),

    desc_arr(sigma_implied_frz, 'LSTM_implied_sigma_from_ret_frozen_test'),
    desc_arr(sigma_implied_wf,  'LSTM_implied_sigma_from_ret_walkfwd_test'),
    desc_arr(walk_gk_vol,       'GK_vol_test'),
    desc_arr(walk_garch,        'GARCH_vol_test'),
])
df_sig_stats.to_csv(os.path.join(OUTPUT_DIR, "implied_sigma_stats.csv"), index=False, encoding='utf-8-sig')
print("✓ Saved implied_sigma_stats.csv")


# -----------------------------------------------------------------------------
# 9.5 逐日 CSV（訓練期）
# -----------------------------------------------------------------------------
df_daily_tr = pd.DataFrame({
    'DATE':                dates_tr,
    'ES1_CLOSE':           close_tr,
    'ES1_LN_RET':          ret_tr,

    # 三模型 VaR_ret
    'gk_VaR_ret_95':       gk_var_ret_train,
    'garch_VaR_ret_95':    garch_var_ret_tr,
    'lstm_VaR_ret_95':     lstm_var_ret_tr,

    # 三模型 VaR_price（由 ret 轉出）
    'gk_VaR_price_95':     gk_var_price_tr,
    'garch_VaR_price_95':  garch_var_price_tr,
    'lstm_VaR_price_95':   lstm_var_price_tr,

    # 波動比較
    'sigma_implied':       sigma_implied_tr,
    'gk_vol_daily':        gk_vol_tr_arr,
    'garch_vol':           garch_vol_tr_arr,

    # return-level violation
    'viol_gk_ret':         viol_gk_tr,
    'viol_garch_ret':      viol_garch_tr,
    'viol_lstm_ret':       viol_lstm_tr,
}).set_index('DATE')

df_daily_tr.to_csv(os.path.join(OUTPUT_DIR, "daily_train.csv"), encoding='utf-8-sig')
print("✓ Saved daily_train.csv")


# -----------------------------------------------------------------------------
# 9.6 逐日 CSV（測試期）
# -----------------------------------------------------------------------------
df_daily_te = pd.DataFrame({
    'DATE':                 walk_dates,
    'ES1_CLOSE':            walk_close,
    'ES1_LN_RET':           walk_ret,

    # 三模型 VaR_ret
    'gk_VaR_ret_95':        gk_var_ret_test,
    'garch_VaR_ret_95':     garch_var_ret_te,
    'lstm_VaR_ret_wf':      lstm_var_ret_wf,
    'lstm_VaR_ret_frz':     lstm_var_ret_frz,

    # 三模型 VaR_price（由 ret 轉出）
    'gk_VaR_price_95':      gk_var_price_te,
    'garch_VaR_price_95':   garch_var_price_te,
    'lstm_VaR_price_wf':    lstm_var_price_wf,
    'lstm_VaR_price_frz':   lstm_var_price_frz,

    # 波動比較
    'sigma_implied_wf':     sigma_implied_wf,
    'sigma_implied_frz':    sigma_implied_frz,
    'gk_vol_daily':         walk_gk_vol,
    'garch_vol':            walk_garch,

    # return-level violation
    'viol_gk_ret':          viol_gk_te,
    'viol_garch_ret':       viol_garch_te,
    'viol_lstm_wf_ret':     viol_lstm_wf,
    'viol_lstm_frz_ret':    viol_lstm_frz,
}).set_index('DATE')

df_daily_te.to_csv(os.path.join(OUTPUT_DIR, "daily_test.csv"), encoding='utf-8-sig')
print("✓ Saved daily_test.csv")


# =============================================================================
# 10.  圖表輸出（ret-target 版本）
# =============================================================================

# ── Fig 1: 訓練損失曲線 ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(history.history['loss'],     lw=1.5, label='Train Loss')
ax.plot(history.history['val_loss'], lw=1.5, ls='--', label='Val Loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Huber Loss')
ax.set_title('Base Model Training Loss (VaR_ret Target)')
ax.legend()
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig1_training_loss.png"), dpi=200, bbox_inches='tight')
plt.close()
print("✓ fig1_training_loss.png")


# ── Fig 2: 主目標變數預測圖（VaR_ret，訓練 & 測試）──────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('LSTM_GK Direct VaR(95%) Return Prediction vs Actual GK VaR Return',
             fontsize=13, fontweight='bold')

for ax, dates, pred, true, title_text in [
    (
        axes[0],
        dates_tr,
        lstm_var_ret_tr,
        gk_var_ret_train,
        f'Train (2006-2021)\nRMSE={rmse_tr:.6f}  MAE={mae_tr:.6f}  Corr={corr_tr:.3f}'
    ),
    (
        axes[1],
        walk_dates,
        lstm_var_ret_wf,
        gk_var_ret_test,
        f'Walk-Fwd (2022-2025)\nRMSE={rmse_wf:.6f}  MAE={mae_wf:.6f}  Corr={corr_wf:.3f}'
    ),
]:
    ax.plot(dates, true, lw=0.8, alpha=0.9, label='GK VaR_ret (actual)')
    ax.plot(dates, pred, lw=0.9, alpha=0.85, label='LSTM_GK Predicted VaR_ret')
    ax.set_title(title_text, fontsize=10)
    ax.set_ylabel('VaR Return')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(3 if ax is axes[0] else 1))

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig2_var_ret_prediction.png"), dpi=200, bbox_inches='tight')
plt.close()
print("✓ fig2_var_ret_prediction.png")


# ── Fig 3: 三模型 VaR_ret 路徑比較圖（核心圖）───────────────────────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Three-Model VaR(95%) Return Comparison: GK / GARCH / LSTM_GK',
             fontsize=13, fontweight='bold')

for ax, dates, realized_ret, gk_v, ga_v, ls_v, title_text in [
    (
        axes[0],
        dates_tr,
        ret_tr,
        gk_var_ret_train,
        garch_var_ret_tr,
        lstm_var_ret_tr,
        'Train (2006-2021)'
    ),
    (
        axes[1],
        walk_dates,
        walk_ret,
        gk_var_ret_test,
        garch_var_ret_te,
        lstm_var_ret_wf,
        'Walk-Forward (2022-2025)'
    ),
]:
    ax.plot(dates, realized_ret, lw=0.6, alpha=0.55, label='Realized ES1 Return')
    ax.plot(dates, gk_v,  lw=1.1, label='GK VaR95 (benchmark)')
    ax.plot(dates, ga_v,  lw=1.0, ls='--', label='GARCH VaR95')
    ax.plot(dates, ls_v,  lw=1.0, ls='-.', label='LSTM_GK VaR95')
    ax.set_title(title_text, fontsize=11)
    ax.set_ylabel('Return')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.25)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(3 if ax is axes[0] else 1))

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig3_three_model_var_ret_path.png"), dpi=200, bbox_inches='tight')
plt.close()
print("✓ fig3_three_model_var_ret_path.png")


# ── Fig 4: 三模型 VaR_price 路徑比較圖（由 ret 轉出，展示用途）──────────────
fig, axes = plt.subplots(2, 1, figsize=(16, 10))
fig.suptitle('Three-Model VaR(95%) Price Comparison (Derived from VaR Returns)',
             fontsize=13, fontweight='bold')

for ax, dates, close_arr, gk_v, ga_v, ls_v, title_text in [
    (
        axes[0],
        dates_tr,
        close_tr,
        gk_var_price_tr,
        garch_var_price_tr,
        lstm_var_price_tr,
        'Train (2006-2021)'
    ),
    (
        axes[1],
        walk_dates,
        walk_close,
        gk_var_price_te,
        garch_var_price_te,
        lstm_var_price_wf,
        'Walk-Forward (2022-2025)'
    ),
]:
    ax.plot(dates, close_arr, lw=0.6, alpha=0.6, label='ES1 Close Price')
    ax.plot(dates, gk_v,  lw=1.1, label='GK VaR95 Price')
    ax.plot(dates, ga_v,  lw=1.0, ls='--', label='GARCH VaR95 Price')
    ax.plot(dates, ls_v,  lw=1.0, ls='-.', label='LSTM_GK VaR95 Price')
    ax.set_title(title_text, fontsize=11)
    ax.set_ylabel('Price (Index Points)')
    ax.legend(fontsize=9)
    ax.grid(alpha=0.25)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(3 if ax is axes[0] else 1))

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig4_three_model_var_price_path.png"), dpi=200, bbox_inches='tight')
plt.close()
print("✓ fig4_three_model_var_price_path.png")


# ── Fig 5: 違規事件圖（三模型，測試期，return-level）──────────────────────
fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
fig.suptitle('VaR(95%) Return Violations — Test Period 2022-2025\n(GK / GARCH / LSTM_GK Walk-Fwd)',
             fontsize=13, fontweight='bold')

dates_te_valid = walk_dates

for ax, viol_arr, var_arr, label in [
    (axes[0], viol_gk_te,    gk_var_ret_test, 'GK Realized'),
    (axes[1], viol_garch_te, garch_var_ret_te, 'GARCH Baseline'),
    (axes[2], viol_lstm_wf,  lstm_var_ret_wf, 'LSTM_GK Walk-Forward'),
]:
    ax.plot(dates_te_valid, walk_ret, lw=0.7, alpha=0.75, label='Realized ES1 Return')
    ax.plot(dates_te_valid, var_arr, lw=1.0, label=f'{label} VaR95')
    vm = viol_arr == 1
    ax.scatter(
        dates_te_valid[vm],
        walk_ret[vm],
        s=15,
        zorder=5,
        label=f'Violation (n={viol_arr.sum()}, {viol_arr.mean()*100:.1f}%)'
    )
    ax.set_ylabel('Return')
    ax.legend(fontsize=8, loc='lower left')
    ax.grid(alpha=0.2)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator())

axes[-1].set_xlabel('Date')
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig5_violations_test_ret.png"), dpi=200, bbox_inches='tight')
plt.close()
print("✓ fig5_violations_test_ret.png")


# ── Fig 6: 隱含波動比較圖（LSTM implied vs GK vs GARCH）────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('Implied Sigma from LSTM VaR Return vs GK / GARCH Volatility',
             fontsize=13, fontweight='bold')

for ax, dates, sig_lstm, sig_gk, sig_garch, title_text in [
    (axes[0], dates_tr,   sigma_implied_tr,  gk_vol_tr_arr,   garch_vol_tr_arr, 'Train (2006-2021)'),
    (axes[1], walk_dates, sigma_implied_wf,  walk_gk_vol,     walk_garch,        'Walk-Fwd (2022-2025)'),
]:
    ax.plot(dates, sig_gk,    lw=0.8, alpha=0.85, label='GK vol (actual)')
    ax.plot(dates, sig_garch, lw=0.8, alpha=0.85, ls='--', label='GARCH vol')
    ax.plot(dates, sig_lstm,  lw=0.9, alpha=0.9,  ls='-.', label='LSTM implied σ')
    ax.set_title(title_text, fontsize=11)
    ax.set_ylabel('Volatility σ')
    ax.legend(fontsize=8)
    ax.grid(alpha=0.25)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(3 if ax is axes[0] else 1))

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig6_implied_sigma.png"), dpi=200, bbox_inches='tight')
plt.close()
print("✓ fig6_implied_sigma.png")


# ── Fig 7: 誤差診斷圖（VaR_ret：Scatter + Quintile Bias + Std Ratio）───────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Prediction Diagnosis: VaR Return\n(Scatter / Quintile Bias / Variance Ratio)',
             fontsize=12, fontweight='bold')

# Scatter
ax = axes[0]
ax.scatter(gk_var_ret_test, lstm_var_ret_wf,  s=5, alpha=0.3, label='Walk-Fwd')
ax.scatter(gk_var_ret_test, lstm_var_ret_frz, s=5, alpha=0.25, label='Frozen')

x_min = min(np.nanmin(gk_var_ret_test), np.nanmin(lstm_var_ret_wf), np.nanmin(lstm_var_ret_frz))
x_max = max(np.nanmax(gk_var_ret_test), np.nanmax(lstm_var_ret_wf), np.nanmax(lstm_var_ret_frz))
ax.plot([x_min, x_max], [x_min, x_max], 'k-', lw=1.2, label='45° (perfect)')
ax.set_xlabel('GK VaR Return (actual)')
ax.set_ylabel('LSTM VaR Return (pred)')
ax.set_title(f'Scatter\nWF Corr={corr_wf:.3f}, Frz Corr={corr_frozen:.3f}', fontsize=9)
ax.legend(fontsize=8)
ax.grid(alpha=0.25)

# Quintile bias
ax = axes[1]
valid_both = ~(np.isnan(gk_var_ret_test) | np.isnan(lstm_var_ret_wf) | np.isnan(lstm_var_ret_frz))
gk_v = gk_var_ret_test[valid_both]
ls_v = lstm_var_ret_wf[valid_both]
fr_v = lstm_var_ret_frz[valid_both]

try:
    q = pd.qcut(gk_v, 5, labels=['Q1','Q2','Q3','Q4','Q5'], duplicates='drop')
    qnames = ['Q1','Q2','Q3','Q4','Q5']
    bias_wf  = [(ls_v[q == qn] - gk_v[q == qn]).mean() for qn in qnames]
    bias_frz = [(fr_v[q == qn] - gk_v[q == qn]).mean() for qn in qnames]
    x5 = np.arange(5)
    ax.bar(x5 - 0.2, bias_wf,  0.35, alpha=0.8, label='Walk-Fwd')
    ax.bar(x5 + 0.2, bias_frz, 0.35, alpha=0.8, label='Frozen')
    ax.axhline(0, color='k', lw=1, ls='--')
    ax.set_xticks(x5)
    ax.set_xticklabels(['Q1\n(lower risk)','Q2','Q3','Q4','Q5\n(higher risk)'])
except Exception:
    ax.text(0.5, 0.5, 'Quintile unavailable', ha='center', transform=ax.transAxes)

ax.set_ylabel('Mean Bias (Pred - GK)')
ax.set_title('Bias by GK-VaR Return Quintile', fontsize=9)
ax.legend(fontsize=8)
ax.grid(axis='y', alpha=0.3)

# Variance suppression
ax = axes[2]
std_ratios = [
    np.nanstd(lstm_var_ret_tr)  / np.nanstd(gk_var_ret_train),
    np.nanstd(lstm_var_ret_frz) / np.nanstd(gk_var_ret_test),
    np.nanstd(lstm_var_ret_wf)  / np.nanstd(gk_var_ret_test),
]
labs = ['Train', 'Frozen', 'Walk-Fwd']
bars = ax.bar(labs, std_ratios, edgecolor='white', width=0.45)
ax.axhline(1.0, color='k', lw=1.5, ls='--', label='Perfect = 1.0')
for bar, v in zip(bars, std_ratios):
    ax.text(bar.get_x() + bar.get_width()/2, v + 0.01, f'{v:.3f}',
            ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('Pred Std / GK Std')
ax.set_title('Variance Ratio\n(<1 = model compresses VaR range)', fontsize=9)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, max(1.4, np.nanmax(std_ratios) * 1.15))

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig7_diagnosis_ret.png"), dpi=200, bbox_inches='tight')
plt.close()
print("✓ fig7_diagnosis_ret.png")


# ── Fig 8: VaR Violation Rate 比較柱狀圖（return-level）────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('VaR(95%) Violation Rate — Return Level\n(GK / GARCH / LSTM_GK)',
             fontsize=13, fontweight='bold')

for ax, rows, title_text in [
    (
        axes[0],
        [
            ('GK\nBenchmark',      k_gk_tr['viol_rate']),
            ('GARCH\nBaseline',    k_ga_tr['viol_rate']),
            ('LSTM_GK\nBase',      k_ls_tr['viol_rate']),
        ],
        'Train (2006-2021)'
    ),
    (
        axes[1],
        [
            ('GK\nBenchmark',      k_gk_te['viol_rate']),
            ('GARCH\nBaseline',    k_ga_te['viol_rate']),
            ('LSTM Frozen\n(A-ver)', k_ls_frz['viol_rate']),
            ('LSTM WF\n(B-ver)',   k_ls_wf['viol_rate']),
        ],
        'Test (2022-2025)'
    ),
]:
    names = [r[0] for r in rows]
    vrs   = [r[1] * 100 for r in rows]
    bars = ax.bar(names, vrs, edgecolor='white', width=0.5)
    ax.axhline(5.0, ls='--', lw=1.5, label='5% target')
    for bar, v in zip(bars, vrs):
        ax.text(bar.get_x() + bar.get_width()/2, v + 0.15,
                f'{v:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')
    ax.set_ylabel('Violation Rate (%)')
    ax.set_title(title_text, fontsize=11)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)
    ax.set_ylim(0, max(vrs) * 1.3 if len(vrs) > 0 else 6)

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig8_violation_rate_ret.png"), dpi=200, bbox_inches='tight')
plt.close()
print("✓ fig8_violation_rate_ret.png")


# ── Fig 9: Annual violation rate（Walk-Fwd 測試期，return-level）────────────
ann = pd.DataFrame({
    'year': walk_dates.year,
    'gk':   viol_gk_te,
    'ga':   viol_garch_te,
    'ls':   viol_lstm_wf
}).groupby('year').mean().reset_index()

fig, ax = plt.subplots(figsize=(11, 5))
fig.suptitle('Annual VaR(95%) Violation Rate — Test Period 2022-2025',
             fontsize=12, fontweight='bold')

x_pos = np.arange(len(ann))
w = 0.25
ax.bar(x_pos - w, ann['gk'] * 100, w, label='GK')
ax.bar(x_pos,     ann['ga'] * 100, w, label='GARCH')
ax.bar(x_pos + w, ann['ls'] * 100, w, label='LSTM_GK WF')
ax.axhline(5.0, ls='--', lw=1.5, label='5% target')
ax.set_xticks(x_pos)
ax.set_xticklabels(ann['year'].astype(str))
ax.set_ylabel('Violation Rate (%)')
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig9_annual_violation_ret.png"), dpi=200, bbox_inches='tight')
plt.close()
print("✓ fig9_annual_violation_ret.png")


# ── Fig 10: Rolling 60-day violation（測試期，return-level）─────────────────
df_roll = pd.DataFrame({
    'gk': viol_gk_te,
    'ga': viol_garch_te,
    'ls': viol_lstm_wf
}, index=walk_dates)

fig, ax = plt.subplots(figsize=(14, 4.5))
ax.plot(walk_dates, df_roll['gk'].rolling(60).mean(), lw=1.5, label='GK')
ax.plot(walk_dates, df_roll['ga'].rolling(60).mean(), lw=1.5, ls='--', label='GARCH')
ax.plot(walk_dates, df_roll['ls'].rolling(60).mean(), lw=1.5, ls='-.', label='LSTM_GK WF')
ax.axhline(0.05, ls=':', lw=1.2, label='5% target')
ax.set_ylabel('Rolling 60d Violation Rate')
ax.set_title('Rolling 60-Day VaR(95%) Violation Rate — Test 2022-2025')
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=6))
plt.xticks(rotation=30)
ax.legend(fontsize=9)
ax.grid(alpha=0.25)

fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig10_rolling_violation_ret.png"), dpi=200, bbox_inches='tight')
plt.close()
print("✓ fig10_rolling_violation_ret.png")

# ── Fig 11: Price-level violation（測試期，補充展示）────────────────────────
# violation 定義：ES1_CLOSE_t < VaR_price_t
viol_gk_price_te    = (walk_close < gk_var_price_te).astype(int)
viol_garch_price_te = (walk_close < garch_var_price_te).astype(int)
viol_lstm_price_wf  = (walk_close < lstm_var_price_wf).astype(int)

fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)
fig.suptitle('VaR(95%) Price-Level Violations — Test Period 2022-2025\n(GK / GARCH / LSTM_GK Walk-Fwd)',
             fontsize=13, fontweight='bold')

for ax, viol_arr, var_arr, label in [
    (axes[0], viol_gk_price_te,    gk_var_price_te,    'GK Realized'),
    (axes[1], viol_garch_price_te, garch_var_price_te, 'GARCH Baseline'),
    (axes[2], viol_lstm_price_wf,  lstm_var_price_wf,  'LSTM_GK Walk-Forward'),
]:
    ax.plot(walk_dates, walk_close, lw=0.7, alpha=0.75, label='ES1 Close')
    ax.plot(walk_dates, var_arr,    lw=1.0, label=f'{label} VaR95 Price')

    vm = viol_arr == 1
    ax.scatter(
        walk_dates[vm],
        walk_close[vm],
        s=15,
        zorder=5,
        label=f'Violation (n={viol_arr.sum()}, {viol_arr.mean()*100:.1f}%)'
    )

    ax.set_ylabel('Price')
    ax.legend(fontsize=8, loc='lower left')
    ax.grid(alpha=0.2)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator())

axes[-1].set_xlabel('Date')
fig.tight_layout()
fig.savefig(os.path.join(OUTPUT_DIR, "fig11_price_level_violations.png"), dpi=200, bbox_inches='tight')
plt.close()
print("✓ fig11_price_level_violations.png")

# =============================================================================
# 11.  最終摘要（ret-target 版本）
# =============================================================================

print("\n" + "="*75)
print("FINAL SUMMARY")
print("="*75)

# -----------------------------------------------------------------------------
# 11.1 Forecast Metrics（VaR_ret）
# -----------------------------------------------------------------------------
print(f"\n[VaR Return Forecast — MAE / RMSE / Corr]")
print(f"  Train (Base)      : MAE={mae_tr:.6f}  RMSE={rmse_tr:.6f}  Corr={corr_tr:.4f}")
print(f"  Frozen A-ver      : MAE={mae_frozen:.6f}  RMSE={rmse_frozen:.6f}  Corr={corr_frozen:.4f}")
print(f"  Walk-Fwd B-ver    : MAE={mae_wf:.6f}  RMSE={rmse_wf:.6f}  Corr={corr_wf:.4f}")

# -----------------------------------------------------------------------------
# 11.2 Implied Sigma Summary
# -----------------------------------------------------------------------------
print(f"\n[Implied Sigma from VaR Return — Test Period]")
print(f"  LSTM implied σ (Frozen) : mean={np.nanmean(sigma_implied_frz):.6f}")
print(f"  LSTM implied σ (WalkFwd): mean={np.nanmean(sigma_implied_wf):.6f}")
print(f"  GK vol daily            : mean={np.nanmean(walk_gk_vol):.6f}")
print(f"  GARCH vol               : mean={np.nanmean(walk_garch):.6f}")

# -----------------------------------------------------------------------------
# 11.3 Return-Level Backtest
# -----------------------------------------------------------------------------
print(f"\n[VaR(95%) Backtest — Return Level]")
for name, k, c in [
    ("GK Realized   (Train)",   k_gk_tr,  c_gk_tr),
    ("GARCH         (Train)",   k_ga_tr,  c_ga_tr),
    ("LSTM_GK Base  (Train)",   k_ls_tr,  c_ls_tr),
    ("GK Realized   (Test)",    k_gk_te,  c_gk_te),
    ("GARCH         (Test)",    k_ga_te,  c_ga_te),
    ("LSTM_GK Frozen(Test)",    k_ls_frz, c_ls_frz),
    ("LSTM_GK WalkFwd(Test)",   k_ls_wf,  c_ls_wf),
]:
    print(
        f"  {name:<24s}: "
        f"Viol={k['x']:4d}/{k['n']:<4d} ({k['viol_rate']*100:5.2f}%)  "
        f"UC={'PASS' if k['p_uc'] >= 0.05 else 'FAIL'}(p={k['p_uc']:.4f})  "
        f"CC={'PASS' if c['p_cc'] >= 0.05 else 'FAIL'}(p={c['p_cc']:.4f})  "
        f"IND(p={c['p_ind']:.4f})"
    )

# -----------------------------------------------------------------------------
# 11.4 Price-Level Backtest（補充展示）
# -----------------------------------------------------------------------------
# 若前面有建立 Fig 11 對應的 price-level violation，可一起摘要
if all(v in globals() for v in [
    'viol_gk_price_te', 'viol_garch_price_te', 'viol_lstm_price_wf'
]):
    print(f"\n[VaR(95%) Backtest — Price Level (Supplementary)]")

    def quick_bt_stats(viol_arr, alpha=ALPHA_95):
        k = kupiec_test(viol_arr, alpha)
        c = christoffersen_cc(viol_arr, alpha)
        return k, c

    k_gk_p,  c_gk_p  = quick_bt_stats(viol_gk_price_te,    ALPHA_95)
    k_ga_p,  c_ga_p  = quick_bt_stats(viol_garch_price_te, ALPHA_95)
    k_ls_p,  c_ls_p  = quick_bt_stats(viol_lstm_price_wf,  ALPHA_95)

    for name, k, c in [
        ("GK Realized   (Test)",  k_gk_p, c_gk_p),
        ("GARCH         (Test)",  k_ga_p, c_ga_p),
        ("LSTM_GK WF    (Test)",  k_ls_p, c_ls_p),
    ]:
        print(
            f"  {name:<24s}: "
            f"Viol={k['x']:4d}/{k['n']:<4d} ({k['viol_rate']*100:5.2f}%)  "
            f"UC={'PASS' if k['p_uc'] >= 0.05 else 'FAIL'}(p={k['p_uc']:.4f})  "
            f"CC={'PASS' if c['p_cc'] >= 0.05 else 'FAIL'}(p={c['p_cc']:.4f})"
        )

# -----------------------------------------------------------------------------
# 11.5 Derived VaR Price Summary
# -----------------------------------------------------------------------------
print(f"\n[Derived VaR Price Summary — Test Period]")
print(f"  GK VaR price mean        : {np.nanmean(gk_var_price_te):.2f}")
print(f"  GARCH VaR price mean     : {np.nanmean(garch_var_price_te):.2f}")
print(f"  LSTM Frozen VaR price    : {np.nanmean(lstm_var_price_frz):.2f}")
print(f"  LSTM WalkFwd VaR price   : {np.nanmean(lstm_var_price_wf):.2f}")

# -----------------------------------------------------------------------------
# 11.6 Output file list
# -----------------------------------------------------------------------------
print(f"\n[Output Directory]")
print(f"  {OUTPUT_DIR}/")

for f in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, f)
    if os.path.isfile(fp):
        print(f"  {f:<45s} {os.path.getsize(fp)//1024:5d} KB")

✓ Saved backtest_summary.csv
✓ Saved forecast_metrics.csv
✓ Saved var_ret_stats.csv
✓ Saved var_price_stats.csv
✓ Saved implied_sigma_stats.csv
✓ Saved daily_train.csv
✓ Saved daily_test.csv
✓ fig1_training_loss.png
✓ fig2_var_ret_prediction.png
✓ fig3_three_model_var_ret_path.png
✓ fig4_three_model_var_price_path.png
✓ fig5_violations_test_ret.png
✓ fig6_implied_sigma.png
✓ fig7_diagnosis_ret.png
✓ fig8_violation_rate_ret.png
✓ fig9_annual_violation_ret.png
✓ fig10_rolling_violation_ret.png
✓ fig11_price_level_violations.png

FINAL SUMMARY

[VaR Return Forecast — MAE / RMSE / Corr]
  Train (Base)      : MAE=0.004314  RMSE=0.006956  Corr=0.8230
  Frozen A-ver      : MAE=0.004587  RMSE=0.007006  Corr=0.7062
  Walk-Fwd B-ver    : MAE=0.004612  RMSE=0.006961  Corr=0.7064

[Implied Sigma from VaR Return — Test Period]
  LSTM implied σ (Frozen) : mean=0.009086
  LSTM implied σ (WalkFwd): mean=0.009303
  GK vol daily            : mean=0.009731
  GARCH vol               : mean=0.010947

[VaR(